# Part-damage classification: real data, then synthetic fine-tuning

## Goal
Train a ResNet18 classifier for each of **mobility**, **structure**, and **mission equipment** using the original human-labeled Humvee boxes. Then continue each model's training on the extra synthetic crops at a lower learning rate. Preserve both stages and compare them on the same real-image validation and test sets.

This is an exploratory military damage baseline initialized from ImageNet, not a civilian-damage transfer experiment. No damage model has been trained by creating this notebook. Source-class names and the four canonical damage levels stay unchanged. `unobservable` is a separate visual category, not a severity above severe damage.

The 30 synthetic images and CVAT annotations from `Desktop/synthetic` are already imported under `datasets/military/damage/source/synthetic_humvee_damage_v1/`. Their image hashes and original XML hash were verified against that folder on September 9, 2026.

**Run order:** inventory and review -> source-group split and crops -> real training -> synthetic fine-tuning -> validation comparison -> final real-image test. Human review is required before crop preparation; the notebook does not silently accept default CVAT labels or provisional source groups.

## Setup
Use a Python kernel with a matching PyTorch/torchvision installation for your CPU or GPU. On a fresh computer, first clone this branch with Git LFS and fetch the images; see [Run on another computer](README.md#run-on-another-computer). From the repository root, create an environment if `.venv` does not exist, then install the notebook dependencies:

```powershell
# On a fresh clone only; reuse an existing project environment if available.
python -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r experiments/damage/requirements.txt
.\.venv\Scripts\python.exe -m ipykernel install --user --name crater --display-name "CRATER"
.\.venv\Scripts\python.exe -m jupyter lab experiments/damage/part_damage_classification.ipynb
```

Select the CRATER kernel. Install the requirements before running the first code cell; they include pandas. For NVIDIA GPU training, install the matching torch/torchvision build from the [PyTorch installer](https://pytorch.org/get-started/locally/) in this environment before the remaining requirements. The notebook prints the selected device. Use one environment consistently; the default Conda Python mixed with user-site PyTorch produced an OpenMP conflict during validation. The default `INITIALIZATION = "imagenet"` downloads official ResNet18 weights on the first training run if they are not cached. Use `"scratch"` for an explicit no-download experiment and a new run name. See the [official torchvision ResNet18 documentation](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html).

All generated crops, review snapshots, checkpoints, metrics and predictions live under ignored `outputs/damage/`. The editable review CSV lives under `datasets/military/damage/review/`; keep a backup of completed human review. Training functions live in `experiments/damage/training.py`, and source preparation lives in `tools/data/prepare_damage_classification.py`.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import json
import sys
from dataclasses import asdict
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "configs/taxonomy.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the CRATER repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageDraw
from IPython.display import display
from tools.data.import_cvat_damage_annotations import sha256_file
from tools.data.prepare_damage_classification import (
    REVIEW_FIELDS, load_inventory, proposed_review_rows, write_csv, read_csv,
    apply_review, materialize_crops,
)
from experiments.damage.training import TrainingConfig, train_stage, final_comparison

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}; device={DEVICE}")

### Parameters
The second stage uses **synthetic training crops only**, starting from the best real-stage checkpoint. A smaller learning rate and shorter schedule limit changes to the learned real-image features. There is no guarantee of improvement; real validation scores decide which checkpoint is retained.

The same seed, labels, crop preprocessing and real validation/test membership apply to both stages. Use a new `RUN_NAME` when changing data, grouping, initialization or training settings. Existing training directories are never overwritten.

In [ ]:
RUN_NAME = "humvee_damage_real_then_synthetic_v1"
RUN_DIR = ROOT / "outputs/damage/humvee_cvat" / RUN_NAME
CROP_DIR = RUN_DIR / "data"
REVIEW_PATH = ROOT / "datasets/military/damage/review/humvee_real_synthetic_v1.csv"
PART_GROUPS = ["mobility", "structure", "mission_equipment"]
LEVELS = json.loads((ROOT / "configs/taxonomy.json").read_text())["damage"]["levels"]
INITIALIZATION = "imagenet"
REAL_CONFIG = TrainingConfig(epochs=20, learning_rate=1e-4, batch_size=32, seed=42)
SYNTHETIC_CONFIG = TrainingConfig(epochs=10, learning_rate=1e-5, batch_size=32, seed=42)
print("Damage label order:", LEVELS)
print("Run directory:", RUN_DIR)

## Steps
### 1. Verify the original and synthetic source inventories
This checks image hashes, image dimensions, manual-label provenance, box geometry and taxonomy. Synthetic generation prompts are never used as labels. Ground-truth boxes remain distinct from the separate detector-crop labeling pilot.

In [ ]:
inventory = load_inventory(ROOT, include_synthetic=True)
inventory_table = pd.DataFrame(inventory)
display(inventory_table.groupby("domain").agg(
    images=("source_image", "nunique"), boxes=("annotation_id", "count")))
display(pd.crosstab([inventory_table.domain, inventory_table.component_group],
                    inventory_table.damage_level).reindex(columns=LEVELS, fill_value=0))

### 2. Review source groups and annotations
The first run creates a CSV with one row per source image and `review_status=pending`. Real `group_id` values are **proposals** from the existing detector pilot's Commons-ID proximity rule and its three manually linked photo series. They are not authoritative scene/vehicle identities. Synthetic group IDs come from recorded generation lineage.

Open the CSV printed below. Inspect the images, merge related photos into the same group, and mark each row `accept` or `reject`. Acceptance means you checked the grouping, part identity, box fit and damage labels, including boxes left at CVAT's default `intact`. For synthetic images, also check generation artifacts. Rejection excludes every box from that image; correct individual labels in CVAT and reimport if needed. Do not assign unique groups merely to make the split pass.

No train/validation/test assignment is made until this review is complete. All derivatives of each reviewed real group inherit one split. Synthetic crops always go into training.

In [ ]:
if not REVIEW_PATH.exists():
    REVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
    write_csv(REVIEW_PATH, proposed_review_rows(inventory), REVIEW_FIELDS)
review_table = pd.DataFrame(read_csv(REVIEW_PATH))
print("Edit this review CSV:", REVIEW_PATH)
display(review_table.groupby(["domain", "review_status"]).size().rename("images").to_frame())
display(review_table.head(8))

### Inspect an annotated source image
Change `REVIEW_IMAGE_INDEX` to inspect another row. The label table retains full annotation IDs for CVAT corrections. Use the original-resolution source image when the resized preview is insufficient.

In [ ]:
REVIEW_IMAGE_INDEX = 0
review_image = review_table.iloc[REVIEW_IMAGE_INDEX]
image_rows = inventory_table[
    (inventory_table.domain == review_image.domain)
    & (inventory_table.source_image == review_image.source_image)
]
source_path = ROOT / image_rows.iloc[0].image_path
with Image.open(source_path) as source:
    preview = source.convert("RGB")
draw = ImageDraw.Draw(preview)
for number, (_, row) in enumerate(image_rows.iterrows()):
    box = tuple(float(row[f"box_{axis}"]) for axis in ("x1", "y1", "x2", "y2"))
    draw.rectangle(box, outline="lime", width=3)
    draw.text((box[0], box[1]), str(number), fill="yellow", stroke_width=2, stroke_fill="black")
preview.thumbnail((1400, 1000))
print(source_path)
display(preview)
display(image_rows[["annotation_id", "component_class", "damage_state_source", "damage_level"]].reset_index(drop=True))

### 3. Split reviewed real source groups and prepare crops
This cell stops with an actionable message while review rows remain pending. It uses a deterministic approximately 70/15/15 split of real source groups, before creating any crops. It does not search seeds for favorable scores or force rare examples across group boundaries.

Inspect the coverage table before training. Sparse classes may be missing from a split. The real source has only three damaged mission-equipment boxes and no unobservable mission-equipment examples; group membership may leave a head without enough training classes. Such a head needs more reviewed data, not an image-random split. Whole annotated boxes are preserved with aspect-ratio padding during training; random center crops could remove the damage itself.

In [ ]:
reviewed = apply_review(inventory, REVIEW_PATH, seed=REAL_CONFIG.seed)
preparation = {
    "review_sha256": sha256_file(REVIEW_PATH),
    "seed": REAL_CONFIG.seed,
    "taxonomy_sha256": sha256_file(ROOT / "configs/taxonomy.json"),
    "sources": sorted({(r["domain"], r["source_labels_sha256"], r["source_manifest_sha256"])
                       for r in inventory}),
}
# JSON normalization makes saved and in-memory tuples/lists comparable.
preparation = json.loads(json.dumps(preparation))
if CROP_DIR.exists():
    metadata_file = CROP_DIR / "preparation.json"
    if not metadata_file.is_file() or json.loads(metadata_file.read_text()) != preparation:
        raise ValueError("Incomplete or changed prepared data. Choose a new RUN_NAME.")
    crop_rows = read_csv(CROP_DIR / "crops_manifest.csv")
else:
    crop_rows = materialize_crops(ROOT, reviewed, CROP_DIR)
    (CROP_DIR / "review_snapshot.csv").write_bytes(REVIEW_PATH.read_bytes())
    (CROP_DIR / "preparation.json").write_text(json.dumps(preparation, indent=2) + "\n")
crop_table = pd.DataFrame(crop_rows)
coverage = pd.crosstab([crop_table.domain, crop_table.split, crop_table.component_group],
                      crop_table.damage_level).reindex(columns=LEVELS, fill_value=0)
display(coverage)
coverage.to_csv(CROP_DIR / "class_coverage.csv")
print("Prepared crops:", len(crop_rows), "Manifest:", CROP_DIR / "crops_manifest.csv")

### 4. Check that the requested heads can run
A head needs at least two training classes and nonempty real validation/test crops. Missing classes remain explicit in the four-output contract; their performance is not established. Do not change groups or seeds after examining model test scores.

In [ ]:
problems = []
for group in PART_GROUPS:
    for domain in ("real", "synthetic"):
        selected = crop_table[(crop_table.component_group == group)
                              & (crop_table.domain == domain) & (crop_table.split == "train")]
        if len(selected) < 2 or selected.damage_level.nunique() < 2:
            problems.append(f"{group}/{domain}: need >=2 training crops and >=2 observed damage classes")
    for split in ("val", "test"):
        selected = crop_table[(crop_table.component_group == group)
                              & (crop_table.domain == "real") & (crop_table.split == split)]
        if selected.empty:
            problems.append(f"{group}: no real {split} crops")
if problems:
    raise ValueError("\n".join(problems) + "\nCollect/review more data or explicitly defer that part group in PART_GROUPS.")
print("Selected part groups have data for both training stages and real evaluation.")

### 5. Train on the original labeled real crops
Run this cell to start training. Each head uses an independent ResNet18 with four outputs. ImageNet initialization supplies general visual features; no trained civilian damage checkpoint currently exists. Training uses inverse-square-root class-frequency loss weights from the current training set and horizontal flips. Batch-normalization running statistics stay fixed to avoid unstable updates from small per-group batches.

Each epoch reports real validation macro F1. The best validation epoch is saved as `real/<group>/best.pt`; `last.pt` preserves the last epoch. Metrics and the complete run configuration are saved alongside them. Test images are not passed to model selection.

In [ ]:
real_checkpoints = {}
for group in PART_GROUPS:
    real_checkpoints[group] = train_stage(
        rows=crop_rows, crop_root=CROP_DIR, output_dir=RUN_DIR / "real" / group,
        group=group, levels=LEVELS, domain="real", config=REAL_CONFIG,
        device=DEVICE, initialization=INITIALIZATION,
    )

### 6. Continue training on the synthetic crops
This stage loads each real-stage `best.pt`, including its learned classifier, and fine-tunes on synthetic crops only. It starts a fresh optimizer at the smaller learning rate. It never overwrites the original checkpoint. A parent-checkpoint hash and the shared data-manifest hash establish the lineage.

The unchanged real model is recorded as synthetic-stage **epoch 0**. If additional synthetic training never improves real validation macro F1, `synthetic/<group>/best.pt` retains that baseline. `last.pt` and `history.json` still preserve the actual synthetic adaptation and any decline. This fallback prevents an assumed synthetic benefit from replacing a better real model.

In [ ]:
synthetic_checkpoints = {}
for group in PART_GROUPS:
    synthetic_checkpoints[group] = train_stage(
        rows=crop_rows, crop_root=CROP_DIR, output_dir=RUN_DIR / "synthetic" / group,
        group=group, levels=LEVELS, domain="synthetic", config=SYNTHETIC_CONFIG,
        device=DEVICE, parent_checkpoint=real_checkpoints[group],
    )

## Checks
### 7. Compare real validation scores and freeze model selection
The primary score is macro F1 over the fixed four-label vocabulary, with zero F1 for unsupported classes. Per-class recall is `null` when a class has no held-out examples; it must not be interpreted as measured zero recall. Accuracy alone would mostly reward intact predictions on this imbalanced source.

Select between stages using validation only. A synthetic best epoch of 0 means the additional training did not earn a replacement. The plot includes every synthetic epoch so any loss of real-image performance is visible.

In [ ]:
comparison_rows, selected_checkpoints = [], {}
fig, axes = plt.subplots(1, len(PART_GROUPS), figsize=(6 * len(PART_GROUPS), 4), squeeze=False)
for axis, group in zip(axes[0], PART_GROUPS):
    real = torch.load(real_checkpoints[group], map_location="cpu", weights_only=True)
    synthetic = torch.load(synthetic_checkpoints[group], map_location="cpu", weights_only=True)
    delta = synthetic["validation"]["macro_f1"] - real["validation"]["macro_f1"]
    selected_checkpoints[group] = synthetic_checkpoints[group] if delta > 0 else real_checkpoints[group]
    comparison_rows.append({"group": group, "real_macro_f1": real["validation"]["macro_f1"],
                            "synthetic_macro_f1": synthetic["validation"]["macro_f1"],
                            "delta": delta, "synthetic_best_epoch": synthetic["epoch"],
                            "selected_stage": "synthetic" if delta > 0 else "real"})
    for stage in ("real", "synthetic"):
        history = json.loads((RUN_DIR / stage / group / "history.json").read_text())
        axis.plot([r["epoch"] for r in history], [r["validation"]["macro_f1"] for r in history], label=stage)
    axis.set(title=group, xlabel="Epoch within stage", ylabel="Real validation macro F1", ylim=(0, 1))
    axis.legend()
plt.tight_layout()
plt.show()
validation_comparison = pd.DataFrame(comparison_rows)
display(validation_comparison)
validation_comparison.to_csv(RUN_DIR / "validation_comparison.csv", index=False)
selection = {group: {"path": str(path.relative_to(RUN_DIR)), "sha256": sha256_file(path)}
             for group, path in selected_checkpoints.items()}
(RUN_DIR / "selected_checkpoints.json").write_text(json.dumps(selection, indent=2) + "\n")

### 8. Evaluate both frozen stages on held-out real images
Run once after all settings and validation-based checkpoint selections are fixed. Synthetic images are excluded. This writes per-crop predictions, confusion matrices, per-class support/recall/F1, macro F1, accuracy, negative log likelihood, multiclass Brier score and ten-bin expected calibration error. These are diagnostic scores, not a fitted confidence calibration or an operational damage-effect validation.

If this cell has already run, it reloads the saved results rather than reevaluating. Do not use test results to choose a different epoch, split or learning rate. The before/after test comparison is a final report for the already selected models.

In [ ]:
TEST_DIR = RUN_DIR / "test_final"
paired_checkpoints = {f"{group}_{stage}": paths[group]
                      for stage, paths in (("real", real_checkpoints), ("synthetic", synthetic_checkpoints))
                      for group in PART_GROUPS}
if TEST_DIR.exists():
    result_file = TEST_DIR / "comparison.json"
    if not result_file.is_file():
        raise ValueError("Final test output is incomplete; inspect it before attempting another evaluation.")
    test_results = json.loads(result_file.read_text())
    if set(test_results) != set(paired_checkpoints) or any(
        test_results[name]["checkpoint_sha256"] != sha256_file(path)
        for name, path in paired_checkpoints.items()
    ):
        raise ValueError("Saved final test results belong to different checkpoints.")
else:
    test_results = final_comparison(paired_checkpoints, crop_rows, CROP_DIR, TEST_DIR, DEVICE)
display(pd.DataFrame([{"model": name, **{key: result[key] for key in
    ("samples", "macro_f1", "accuracy", "ece_10_bins", "brier_score")}}
    for name, result in test_results.items()]))

### 9. Inspect confusion and class support
Rows are true labels and columns are predicted labels. A blank class population is a data-coverage gap. Inspect the saved prediction CSVs by `component_class` to distinguish, for example, damaged weapon stations from communications equipment within the same broad head.

In [ ]:
MODEL_TO_INSPECT = f"{PART_GROUPS[0]}_synthetic"
result = test_results[MODEL_TO_INSPECT]
display(pd.DataFrame(result["per_class"]).T)
fig, axis = plt.subplots(figsize=(8, 6))
confusion = result["confusion_matrix"]
plot = axis.imshow(confusion, cmap="Blues")
axis.set(xticks=range(len(LEVELS)), yticks=range(len(LEVELS)),
         xticklabels=LEVELS, yticklabels=LEVELS, xlabel="Predicted label", ylabel="True label",
         title=f"Held-out real crops: {MODEL_TO_INSPECT}")
plt.setp(axis.get_xticklabels(), rotation=35, ha="right")
for i, row in enumerate(confusion):
    for j, count in enumerate(row):
        axis.text(j, i, str(count), ha="center", va="center")
fig.colorbar(plot, ax=axis, label="Crops")
plt.tight_layout()
plt.show()

## Next steps
- Record whether the synthetic stage improves real validation and final test performance, using the saved scores rather than assuming an improvement.
- Review errors and class coverage, especially the sparse mission-equipment damage labels. Additional independent real scenes are needed for stronger evidence.
- Evaluate detector-predicted crops separately before connecting the classifier to the detector. These results measure classification of human-box crops and do not include detector misses or localization errors.
- Do not interpret softmax confidence as calibrated reliability. `unobservable` is a learned category; a separately validated abstention policy and operational-effect reasoning remain future work.

**Execution status at creation:** the notebook is provided without training outputs. All cells executed successfully on isolated fixture data, and all 13 classifier/annotation tests passed. Production inventory and image-review cells also executed and stopped at the expected pending-review gate. Fixture results are not damage-model performance evidence. Complete the review CSV before running crop preparation and training.